# Clear old vertexnetworking images from R2

Deletes everything under the `products/` folder (the old vertexnetworking images) in your bucket `syntexserver-2003`.

**Safe:** the `dtsupply-products/` folder is a completely separate prefix and will NOT be touched by this.

**This is permanent** — deleted objects can't be recovered. Only run this if you're sure you don't need those old images anymore.

In [1]:
!pip install boto3

In [ ]:
#!/usr/bin/env python3
"""
Deletes all objects under the "products/" prefix in your R2 bucket — this
is where the OLD vertexnetworking images were uploaded to. The
"dtsupply-products/" folder (used for the new site) is NOT touched.

Requirements:
    pip install boto3

Usage:
    python clear_old_vertexnetworking_images.py
"""

import boto3
from botocore.config import Config

# ---------------------------------------------------------------------------
CLOUDFLARE_ACCOUNT_ID = "d42f7c5ed83f1403699b96fc13759c01"
R2_ACCESS_KEY_ID = "93510dc42d0c272e24a95c87f7a20313"
R2_SECRET_ACCESS_KEY = "38de75f0cdb543d0e6f81811faab924e31ec6116cc95edb5d6e5e094a7b11241"
R2_BUCKET_NAME = "syntexserver-2003"

# Only objects under this prefix get deleted. The dtsupply-products/ folder
# is a different prefix and will NOT be touched by this script.
PREFIX_TO_DELETE = "products/"
# ---------------------------------------------------------------------------

s3 = boto3.client(
    "s3",
    endpoint_url=f"https://{CLOUDFLARE_ACCOUNT_ID}.r2.cloudflarestorage.com",
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    config=Config(signature_version="s3v4"),
    region_name="auto",
)


def main():
    print(f"Deleting all objects under '{PREFIX_TO_DELETE}' in bucket '{R2_BUCKET_NAME}'...")
    print("(the 'dtsupply-products/' folder will NOT be touched)\n")

    deleted_total = 0
    continuation_token = None

    while True:
        list_kwargs = {"Bucket": R2_BUCKET_NAME, "Prefix": PREFIX_TO_DELETE}
        if continuation_token:
            list_kwargs["ContinuationToken"] = continuation_token

        response = s3.list_objects_v2(**list_kwargs)
        contents = response.get("Contents", [])

        if not contents:
            break

        # delete_objects accepts up to 1000 keys per call
        objects_to_delete = [{"Key": obj["Key"]} for obj in contents]
        delete_response = s3.delete_objects(
            Bucket=R2_BUCKET_NAME,
            Delete={"Objects": objects_to_delete, "Quiet": True},
        )

        errors = delete_response.get("Errors", [])
        if errors:
            print(f"  [warn] {len(errors)} objects failed to delete in this batch")
            for err in errors[:5]:
                print(f"    {err.get('Key')}: {err.get('Message')}")

        deleted_total += len(objects_to_delete) - len(errors)
        print(f"  Deleted {deleted_total} objects so far...")

        if response.get("IsTruncated"):
            continuation_token = response.get("NextContinuationToken")
        else:
            break

    print(f"\nDone. {deleted_total} objects deleted from '{PREFIX_TO_DELETE}'.")


if __name__ == "__main__":
    main()


Deleting all objects under 'products/' in bucket 'syntexserver-2003'...
(the 'dtsupply-products/' folder will NOT be touched)

  Deleted 1000 objects so far...
  Deleted 2000 objects so far...
  Deleted 3000 objects so far...
  Deleted 4000 objects so far...
  Deleted 5000 objects so far...
  Deleted 6000 objects so far...
  Deleted 7000 objects so far...
  Deleted 8000 objects so far...
  Deleted 9000 objects so far...
  Deleted 10000 objects so far...
  Deleted 11000 objects so far...
  Deleted 12000 objects so far...
  Deleted 13000 objects so far...
  Deleted 14000 objects so far...
  Deleted 15000 objects so far...
  Deleted 16000 objects so far...
  Deleted 17000 objects so far...
  Deleted 18000 objects so far...
  Deleted 19000 objects so far...
  Deleted 20000 objects so far...
  Deleted 21000 objects so far...
